# J-линзы на чистом тексте WikiText

Эталонное обучение J-линзы (`jlens.fit`): 1000 промптов WikiText, до 128 токенов.

- **Llama-3.1-8B-Instruct** → `llama_jacobian_lens_1000.pt` — J-линза Llama, используемая в статье.
- **LLaDA-8B-Instruct** → `llada_jacobian_lens_1000.pt` — линза LLaDA на чистом тексте; в статье не используется и оставлена для сравнения.

Для Llama нужен `HF_TOKEN` с доступом к модели.

In [1]:
import torch
import transformers

import jlens
jlens.configure_logging()

from jlens import Layout
from jlens.examples import load_wikitext_prompts

import os
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in the environment: meta-llama/Llama-3.1-8B-Instruct is a gated model"

In [2]:
import numpy as np
import random

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
device = "cuda:0"

# LLaDA-8B-Instruct

In [4]:
model_name = "GSAI-ML/LLaDA-8B-Instruct"

In [5]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, trust_remote_code=True, dtype=torch.bfloat16, use_cache="./",
).to(device)

hf_model

Loading checkpoint shards: 100%|██████████| 6/6 [00:00<00:00, 222.76it/s]


LLaDAModelLM(
  (model): LLaDAModel(
    (transformer): ModuleDict(
      (wte): Embedding(126464, 4096)
      (emb_drop): Dropout(p=0.0, inplace=False)
      (ln_f): RMSLayerNorm()
      (blocks): ModuleList(
        (0-31): 32 x LLaDALlamaBlock(
          (dropout): Dropout(p=0.0, inplace=False)
          (act): SiLU()
          (attn_out): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_out): Linear(in_features=12288, out_features=4096, bias=False)
          (rotary_emb): RotaryEmbedding()
          (attn_norm): RMSLayerNorm()
          (ff_norm): RMSLayerNorm()
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        )
      )
    

In [6]:
class WrappedLLaDA:
    def __init__(self, model):
        self.model = model
        self.blocks = model.transformer.blocks
        self.ln_f = model.transformer.ln_f
        self.wte = model.transformer.wte

    def forward(self, input_ids, **kwargs):
        return self.model(input_ids, **kwargs)

    def __call__(self, input_ids, **kwargs):
        return self.forward(input_ids, **kwargs)

hf_model.text_module = WrappedLLaDA(hf_model.model)
hf_model.lm_head = hf_model.model.transformer.ff_out

layout = Layout(
    path="text_module",
    layers="blocks",
    norm="ln_f",
    embed="wte",
    lm_head="lm_head"
)

model = jlens.from_hf(hf_model, tokenizer, layout=layout)
model

HFLensModel(LLaDAModelLM, n_layers=32, d_model=4096)

In [7]:
prompts = load_wikitext_prompts(n_prompts=1000)

In [ ]:
lens = jlens.fit(
    model, prompts, dim_batch=32, max_seq_len=128, checkpoint_path="./llada_ckpt_1000.pt"
)
lens.save("./llada_jacobian_lens_1000.pt")

[  1m15s + 75.02s] fit: n_layers=32 d_model=4096, fitting 31 source layers (target=L31) on 1000 prompts
[  1m39s + 24.94s]   prompt 1/1000  seq_len=128 n_valid=111  25s  max||J||/sqrt(d)=1.764  max_d_mean=nan
[  2m06s + 26.85s]   prompt 2/1000  seq_len=128 n_valid=111  25s  max||J||/sqrt(d)=3.476  max_d_mean=1.42e+00
[  2m32s + 26.01s]   prompt 3/1000  seq_len=128 n_valid=111  24s  max||J||/sqrt(d)=2.008  max_d_mean=5.20e-01
[  2m59s + 26.54s]   prompt 4/1000  seq_len=128 n_valid=111  24s  max||J||/sqrt(d)=1.784  max_d_mean=3.10e-01
[  3m26s + 26.71s]   prompt 5/1000  seq_len=128 n_valid=111  24s  max||J||/sqrt(d)=1.994  max_d_mean=3.28e-01
[  4m01s + 35.67s]   prompt 6/1000  seq_len=128 n_valid=111  33s  max||J||/sqrt(d)=6.112  max_d_mean=1.07e+00
[  4m29s + 28.03s]   prompt 7/1000  seq_len=128 n_valid=111  25s  max||J||/sqrt(d)=3.927  max_d_mean=5.44e-01
[  4m56s + 26.45s]   prompt 8/1000  seq_len=128 n_valid=111  24s  max||J||/sqrt(d)=3.650  max_d_mean=3.74e-01
[  5m22s + 26.77s]   

# Llama-3.1-8B-Instruct

In [4]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"

In [5]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, trust_remote_code=True, dtype=torch.bfloat16, use_cache="./",
).to(device)

hf_model

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 156.34it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [6]:
model = jlens.from_hf(hf_model, tokenizer)
model

HFLensModel(LlamaForCausalLM, n_layers=32, d_model=4096)

In [7]:
prompts = load_wikitext_prompts(n_prompts=1000)
lens = jlens.fit(
    model, prompts, dim_batch=32, max_seq_len=128, checkpoint_path="./llama_ckpt_1000.pt"
)
lens.save("./llama_jacobian_lens_1000.pt")

[  1m23s + 83.02s] fit: n_layers=32 d_model=4096, fitting 31 source layers (target=L31) on 1000 prompts
[  1m45s + 22.66s]   prompt 1/1000  seq_len=128 n_valid=111  22s  max||J||/sqrt(d)=6.822  max_d_mean=nan
[  2m10s + 25.31s]   prompt 2/1000  seq_len=128 n_valid=111  23s  max||J||/sqrt(d)=5.867  max_d_mean=6.16e-01
[  2m36s + 25.18s]   prompt 3/1000  seq_len=128 n_valid=111  23s  max||J||/sqrt(d)=9.658  max_d_mean=6.74e-01
[  3m01s + 25.81s]   prompt 4/1000  seq_len=128 n_valid=111  23s  max||J||/sqrt(d)=14.812  max_d_mean=7.95e-01
[  3m27s + 25.54s]   prompt 5/1000  seq_len=128 n_valid=111  23s  max||J||/sqrt(d)=10.615  max_d_mean=4.31e-01
[  3m52s + 25.42s]   prompt 6/1000  seq_len=128 n_valid=111  23s  max||J||/sqrt(d)=9.114  max_d_mean=3.19e-01
[  4m18s + 25.35s]   prompt 7/1000  seq_len=128 n_valid=111  23s  max||J||/sqrt(d)=7.758  max_d_mean=2.61e-01
[  4m43s + 25.71s]   prompt 8/1000  seq_len=128 n_valid=111  23s  max||J||/sqrt(d)=5.155  max_d_mean=1.76e-01
[  5m09s + 25.70s] 